#Telecom Domain ReadOps Assignment
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create schema if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;

In [0]:
dbutils.fs.mkdirs("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

##2. Filesystem operations
1. Write code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2")

In [0]:
customer_csv = ''' 
101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID 
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID 
105,,27,Delhi,PREPAID 
106,Sneha,abc,Pune,PREPAID '''

dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",customer_csv,overwrite=True)

df1 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw",sep=",",header=True,inferSchema=True).toDF("cust_id","cust_name","age","city","plan")
df1.show()


In [0]:
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 
101\t320\t1500\t20 
102\t120\t4000\t5 
103\t540\t600\t52 
104\t45\t200\t2 
105\t0\t0\t0 '''

dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",usage_tsv,overwrite=True)

df2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",sep="\t",header=True,inferSchema=True)
df2.show()


In [0]:
tower_logs_region1 = '''
event_id|customer_id|tower_id|signal_strength|timestamp 
5001|101|TWR01|-80|2025-01-10 10:21:54'''
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",tower_logs_region1,overwrite=True)

df3 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",sep="|",header=True,inferSchema=True)
df3.show()

In [0]:
tower_logs_region2 = '''
event_id|customer_id|tower_id|signal_strength|timestamp 
5004|104|TWR05|-75|2025-01-10 11:01:12 '''
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv",tower_logs_region2,overwrite=True)

df4 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv",sep="|",header=True,inferSchema=True)
df4.show()

##3. Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#Read all tower logs using: Path glob filter (example: *.csv) Multiple paths input Recursive lookup
df1 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",
           "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv"] ,sep = "|")

df1.show()

In [0]:
#Demonstrate these 3 reads separately: Using pathGlobFilter Using list of paths in spark.read.csv([path1, path2]) Using .option("recursiveFileLookup","true")

print("Mulitple Path Read")
df1 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",
           "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv"] ,sep = "|")

df1.show()

print("Multi File in child folders under the same parent path")
df2 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/reg*",sep = "|")

df2.show()

print("Multi File in child folders under the same parent path using pathGlobFilter")
df3 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .option("pathGlobFilter","*.csv")\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/" ,sep = "|")

df3.show()



##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
#Try the Customer, Usage files with the option and options using read.csv and format function:
#header=false, inferSchema=false or header=true, inferSchema=true

df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv")
print("header,inferschema = True")
df.show()


df1 = spark.read.format("csv")\
    .option("header",False)\
    .option("inferSchema",False)\
    .load("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv")
print("header,inferschema = False")
df1.show()

######Write a note on What changed when we use header or inferSchema with true/false?
  As I already loaded both files with inference property enabled, all columns are already converted to string data type and not facing any issues.

######How schema inference handled “abc” in age?
  Age column is conidered as "string" while using inference property.
  


##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
#Apply column names using string using toDF function for customer data
df = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv").toDF("cust_id","cust_name","age","location","plan")
df.show()

In [0]:
#Apply column names and datatype using the schema function for usage data

struct_schema = "cust_id int, cust_name string, age int, location string, plan string"
df = spark.read.schema(struct_schema).csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",sep=',',inferSchema=True)
df.show()

In [0]:
#Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data

from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType

cust_schema = StructType([StructField("event_id",IntegerType(),True),
                      StructField("cust_id",StringType(),True),
                      StructField("tower_id",IntegerType(),True),
                      StructField("signal_strength",IntegerType(),True),
                      StructField("timestamp",TimestampType(),True)])
df = spark.read.schema(cust_schema)\
    .format("csv")\
    .option("pathGlobFilter","*.csv")\
    .option("header","True")\
    .option("recursiveFileLookup","True")\
    .option("delimiter","|")\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")
df.show()

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options


##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#1. Write customer data into CSV format using overwrite mode

df = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv").toDF("cust_id","cust_name","age","location","plan")

df.write.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_csv",mode = "overwrite",header = True)


#2. Write usage data into CSV format using append mode

df1 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",sep = '\t',header = True).toDF("cust_id","voice_min","data_mb","sms_count")
df1.write.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_csv",mode = "overwrite",header = True)

#3. Write tower data into CSV format with header enabled and custom separator (|)

df2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",sep = '|',header = True).toDF("event_id","customer_id","tower_id","signal_strength","timestamp")

df2.write.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_raw_target_csv",mode = "overwrite",header = True,sep = '|')

df3 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv",sep = '|',header = True).toDF("event_id","customer_id","tower_id","signal_strength","timestamp")

df3.write.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_raw_target_csv",mode = "overwrite",header = True,sep = '|')

In [0]:
#7. Read the tower data in a dataframe and show only 5 rows.
df_read = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower*",sep = '|',header = True,recursiveFileLookup=True,pathGlobFilter="*.csv")
df_read.show(5) 

##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.



In [0]:
#1.Write customer data into JSON format using overwrite mode
dfjson = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",header = True, inferSchema = True).toDF("cust_id","cust_name","age","location","plan")

dfjson.show()

dfjson.write.json("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_json",mode = "overwrite")


#2. Write usage data into JSON format using append mode and snappy compression format

dfjson2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",sep = '\t',header = True).toDF("cust_id","voice_min","data_mb","sms_count")
dfjson2.show(2)
dfjson2.write.json("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_json",mode = "append",compression="snappy")


In [0]:
#3.Write tower data into JSON format using ignore mode and observe the behavior of this mode

df3json = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower*",sep = '|',header = True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter="*.csv")
df3json.show(5)

df3json.write.json("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_raw_target_json",mode = "ignore")


#4. Read the tower data in a dataframe and show only 5 rows.

df_read = spark.read.json("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower*",recursiveFileLookup=True,pathGlobFilter="*.json")
df_read.show(5) 



%md
##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#1. Write customer data into Parquet format using overwrite mode and in a gzip format
dfparquet = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",header = True,inferSchema=True,sep = ',')
dfparquet.show(2)

dfparquet.write.parquet("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_parquet",mode = "overwrite",compression="gzip")

#2. Write usage data into Parquet format using error mode

dfparquet2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",header = True,inferSchema=True,sep = '\t')
dfparquet2.show(2)

dfparquet2.write.parquet("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_parquet",mode = "ignore")

#3. Write tower data into Parquet format with gzip compression option
dfparquat3 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower*",sep = '|',header = True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter="*.csv")
dfparquat3.show(5)

dfparquat3.write.parquet("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_raw_target_parquet",mode = "overwrite",compression="gzip")

#4. Read the usage data in a dataframe and show only 5 rows.

df4 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",header = True,inferSchema=True,sep = '\t')
df4.show(5)

%md
##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.


In [0]:
#1. Write customer data into ORC format using overwrite mode
dforc1 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",header = True,inferSchema=True,sep = ',')
dforc1.show(2)

dforc1.write.orc("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_orc",mode = "overwrite")

#2. Write usage data into ORC format using append mode
dforc2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower",header = True,inferSchema=True,sep = '|',recursiveFileLookup=True,pathGlobFilter="*.csv")
dforc2.show(2)      
dforc2.write.orc("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_raw_target_orc",mode = "overwrite")

#3.Write usage data into ORC format using append mode
dforc3 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",header = True,inferSchema=True,sep = '\t')    
dforc3.show(2)
dforc3.write.orc("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_orc",mode = "append")

#4. Read the usage data in a dataframe and show only 5 rows.
dforc4 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",header = True,inferSchema=True,sep = '\t')
dforc4.show(5)
  

%md
##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

In [0]:
#1.Write customer data into Delta format using overwrite mode

from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType,TimestampType

custom_schema = StructType([StructField("cust_id",IntegerType(),True),
                            StructField("cust_name",StringType(),True),
                            StructField("Age",IntegerType(),True),
                            StructField("Location",StringType(),True),
                            StructField("Plan",StringType(),True)])

rfdelta = spark.read.schema(custom_schema).csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",header = True,inferSchema=True,sep = ',')
#rfdelta.show(2)

rfdelta.write.mode("overwrite").format("delta").save("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_delta")

#2. Write usage data into Delta format using append mode

custom_usage_schema = StructType([StructField("cust_id", IntegerType(), True),
                                  StructField("voice_mins",IntegerType(),True),
                                  StructField("data_mb",IntegerType(),True),
                                  StructField("sms_count",DoubleType(),True)])

rfdelta4 = spark.read.format("csv").schema(custom_usage_schema).load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv", sep = '\t',header = True)

display(rfdelta4)

rfdelta4.write.format("delta").mode("overwrite").save("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_delta")

#3.Write tower data into Delta format and see the output file structure

custom_tower_schema = StructType([StructField("event_id",IntegerType(),True),
                                  StructField("customer_id",IntegerType(),True),
                                  StructField("tower_id",StringType(),True),
                                  StructField("signal_strength",IntegerType(),True),
                                  StructField("timestamp",TimestampType(),True)])
                                
dftower5 = spark.read.format("csv").schema(custom_tower_schema).load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower",sep = '|',header = "True",recursiveFileLookup=True,pathGlobFilter="*.csv")

display(dftower5)

dftower5.write.format("delta").mode("overwrite").save("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_raw_target_delta")

#4 Read the usage data in a dataframe and show only 5 rows.
custom_usage_schema = StructType([StructField("cust_id", IntegerType(), True),
                                  StructField("voice_mins",IntegerType(),True),
                                  StructField("data_mb",IntegerType(),True),
                                  StructField("sms_count",DoubleType(),True)])

rfdelta4 = spark.read.format("csv").schema(custom_usage_schema).load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv", sep = '\t',header = True)

display(rfdelta4)

#5 Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
File can be opened in notepad++. However, data is not readable. I can find additional folders in delta log folder and subsequent json and crc files.  


##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


In [0]:
#1. Write customer data using saveAsTable() as a managed table

df_delta_read =spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_delta")
display(df_delta_read)

df_delta_read.write.saveAsTable("telecom_catalog_assign.landing_zone.customer_delta",mode = "overwrite")

#2. Write usage data using saveAsTable() with overwrite mode
df_usage = spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_delta")

display(df_usage)

df_usage.write.saveAsTable("telecom_catalog_assign.landing_zone.usage_table",mode = "overwrite")

#3. Drop the managed table and verify data removal
#%sql
#drop table telecom_catalog_assign.landing_zone.customer_delta




In [0]:

#4. Go and check the table overview and realize it is in delta format in the Catalog.
#Yes. Table properly mentioned as delta in table overview
#5. Use spark.read.sql to write some simple queries on the above tables created.
df_cust1 = spark.sql( "select * from telecom_catalog_assign.landing_zone.customer_delta" )
display(df_cust1)
df_usage2 = spark.sql("select * from telecom_catalog_assign.landing_zone.usage_table")
display(df_usage2)

%md
##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

In [0]:
#1. Write customer data using insertInto() in a new table and find the behavior
df_cust = spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_delta")

display(df_cust)

df_temp = spark.createDataFrame([(107, "Senthilkumar",40,"Chennai","PREPAID"), (108, "Akshaykumar",20,"Madurai","POSTPAID"), (109, "kannan",35,"Thirchy","PREPAID")],schema=["cust_id","cust_name","age","location","plan"])

df_temp.write.insertInto("telecom_catalog_assign.landing_zone.customer_delta")

#2. Write usage data using insertTable() with overwrite mode
df_usage = spark.sql( "select * from telecom_catalog_assign.landing_zone.usage_table" )
display(df_usage)

df_usage1 = spark.createDataFrame([(106,450,1000,12),(107,239,8910,34)],schema=["cust_id","voice_mins","data_mb","sms_count"])
display(df_usage1)

df_usage1.write.insertInto("telecom_catalog_assign.landing_zone.usage_table")



%md
##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

In [0]:
#1. Write customer data into XML format using rowTag as cust
df_cust1 = spark.sql( "select * from telecom_catalog_assign.landing_zone.customer_delta" )
display(df_cust1)

df_cust1.write.format("xml").mode("overwrite").option("rowTag","customer").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw_target_xml")

#2. Write usage data into XML format using overwrite mode with the rowTag as usage
df_usage = spark.sql( "select * from telecom_catalog_assign.landing_zone.usage_table" )
display(df_usage)

df_usage.write.format("xml").mode("overwrite").option("rowTag","usage").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw_target_xml")

#3. Download the xml data and open the file in notepad++ and see how the xml file looks like.
# Yes. Downloaeded and read xml in notepad++. Row tags were created accordingly.


%md
##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType

custom_schema = StructType([StructField("index",IntegerType(),True),StructField("userid",StringType(),True),StructField("firstname",StringType(),True),StructField("lastname",StringType(),True),StructField("sex",StringType(),True),StructField("email",StringType(),True),StructField("phone",StringType(),True),StructField("dateofbirth",TimestampType(),True),StructField("jobtitle",StringType(),True)])

df_csv = spark.read.format("csv").schema(custom_schema).options(header = True,inferschema = True).load("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/Dataset.csv")

display(df_csv.take(10))

df_csv.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/csv_target",mode = "overwrite",header = True)

df_csv.write.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/json_target",mode = "overwrite")

df_csv.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/orc_target",mode = "overwrite")

df_csv.write.parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/parquet_target",mode = "overwrite")

df_csv.write.format("delta").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Large dataset/delta_target", mode= "overwrite", compression = "Gzip")
